In [ ]:
import os
import sys
import math
import sqlite3
from typing import Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rosidl_runtime_py.utilities import get_message
from rclpy.serialization import deserialize_message

# --- SQLite helpers ---
def connect(sqlite_file: str):
    conn = sqlite3.connect(sqlite_file)
    c = conn.cursor()
    return conn, c

def close(conn):
    conn.close()

def get_topics(cursor):
    rows = cursor.execute("SELECT id, name, type FROM topics").fetchall()
    # rows: List[Tuple[id:int, name:str, type:str]]
    return rows

def list_topics(bag_path: str, print_out: bool = True):
    if not os.path.exists(bag_path):
        raise FileNotFoundError(f"Bag file not found: {bag_path}")
    conn, c = connect(bag_path)
    try:
        topics = get_topics(c)
        if print_out:
            print("Topics in bag:")
            for tid, name, typ in topics:
                print(f"- {name} [{typ}] (id={tid})")
        return topics
    finally:
        close(conn)

def find_odom_topic(cursor, color: str = "blue"):
    topics = get_topics(cursor)
    # Case-insensitive match for color tag in topic name
    cand = [(tid, name, typ) for (tid, name, typ) in topics
            if typ == "nav_msgs/msg/Odometry" and (color.lower() in name.lower())]
    if cand:
        return cand[0]  # (id, name, type)
    # fallback to any odom topic
    any_odom = [(tid, name, typ) for (tid, name, typ) in topics
                if typ == "nav_msgs/msg/Odometry"]
    if any_odom:
        return any_odom[0]
    return None

# --- math helpers ---
def yaw_from_quat(qx, qy, qz, qw):
    # yaw (Z) from quaternion
    siny_cosp = 2.0 * (qw * qz + qx * qy)
    cosy_cosp = 1.0 - 2.0 * (qy * qy + qz * qz)
    return math.atan2(siny_cosp, cosy_cosp)

# --- bag reading ---
def read_odom_df(bag_path: str, topic_name: Optional[str] = None, color: str = "blue") -> Tuple[pd.DataFrame, str]:
    if not os.path.exists(bag_path):
        raise FileNotFoundError(f"Bag file not found: {bag_path}")

    conn, c = connect(bag_path)
    try:
        topics = get_topics(c)
        topics_df = pd.DataFrame(topics, columns=["id", "name", "type"])

        if topic_name is None:
            picked = find_odom_topic(c, color=color)
            if picked is None:
                avail = "\n".join([f"- {n} [{t}]" for (_, n, t) in topics])
                raise RuntimeError(
                    f"No nav_msgs/msg/Odometry topic found.\nAvailable topics:\n{avail}"
                )
            topic_id, topic_name, msg_type_name = picked
        else:
            row = topics_df[topics_df["name"] == topic_name]
            if row.empty:
                avail = "\n".join([f"- {n} [{t}]" for (_, n, t) in topics])
                raise RuntimeError(
                    f"Topic '{topic_name}' not found in bag.\nAvailable topics:\n{avail}"
                )
            topic_id = int(row["id"].iloc[0])
            msg_type_name = str(row["type"].iloc[0])

        # resolve ROS 2 message class
        msg_cls = get_message(msg_type_name)

        # stream messages for this topic id (ordered by time)
        q = "SELECT timestamp, data FROM messages WHERE topic_id = ? ORDER BY timestamp ASC"
        rows = c.execute(q, (topic_id,)).fetchall()

        if not rows:
            raise RuntimeError(f"No messages for topic '{topic_name}' in {bag_path}")

        ts = []
        xs, ys, yaws = [], [], []

        for t_ns, blob in rows:
            msg = deserialize_message(blob, msg_cls)
            # Odometry: pose.pose.position, pose.pose.orientation
            p = msg.pose.pose.position
            q = msg.pose.pose.orientation
            xs.append(p.x)
            ys.append(p.y)
            yaws.append(yaw_from_quat(q.x, q.y, q.z, q.w))
            ts.append(t_ns)

        ts = np.asarray(ts, dtype=np.int64)
        t0 = float(ts[0])
        t_sec = (ts - t0) / 1e9

        df = pd.DataFrame(dict(t=t_sec, x=xs, y=ys, yaw=yaws))
        return df, topic_name
    finally:
        close(conn)

# --- plotting ---
def plot_side_by_side(df_ideal: pd.DataFrame, df_noisy: pd.DataFrame,
                      title_left: str, title_right: str, out_path: Optional[str] = None):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), constrained_layout=True)

    # Common axis limits
    xmin = min(df_ideal["x"].min(), df_noisy["x"].min())
    xmax = max(df_ideal["x"].max(), df_noisy["x"].max())
    ymin = min(df_ideal["y"].min(), df_noisy["y"].min())
    ymax = max(df_ideal["y"].max(), df_noisy["y"].max())
    dx = xmax - xmin
    dy = ymax - ymin
    pad_x = 0.05 * (dx if dx > 0 else 1.0)
    pad_y = 0.05 * (dy if dy > 0 else 1.0)

    # Ideal
    ax = axes[0]
    ax.plot(df_ideal["x"], df_ideal["y"], color="tab:blue", lw=1.5)
    ax.scatter(df_ideal["x"].iloc[0], df_ideal["y"].iloc[0], c="green", s=30, label="start")
    ax.scatter(df_ideal["x"].iloc[-1], df_ideal["y"].iloc[-1], c="red", s=30, label="end")
    ax.set_title(title_left)
    ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]")
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(xmin - pad_x, xmax + pad_x)
    ax.set_ylim(ymin - pad_y, ymax + pad_y)
    ax.legend(loc="best")

    # Noisy
    ax = axes[1]
    ax.plot(df_noisy["x"], df_noisy["y"], color="tab:orange", lw=1.5)
    ax.scatter(df_noisy["x"].iloc[0], df_noisy["y"].iloc[0], c="green", s=30, label="start")
    ax.scatter(df_noisy["x"].iloc[-1], df_noisy["y"].iloc[-1], c="red", s=30, label="end")
    ax.set_title(title_right)
    ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]")
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(xmin - pad_x, xmax + pad_x)
    ax.set_ylim(ymin - pad_y, ymax + pad_y)
    ax.legend(loc="best")

    if out_path:
        os.makedirs(os.path.dirname(out_path), exist_ok=True)
        plt.savefig(out_path, dpi=150)
        print(f"Saved figure to {out_path}")
    plt.show()

def plot_blue_from_bags(ideal_bag: str,
                        noisy_bag: str,
                        topic: Optional[str] = None,
                        color: str = "blue",
                        out_path: Optional[str] = None):
    df_i, topic_i = read_odom_df(ideal_bag, topic_name=topic, color=color)
    df_n, topic_n = read_odom_df(noisy_bag, topic_name=topic, color=color)
    left_title = f"Ideal: {topic_i}"
    right_title = f"Noisy: {topic_n}"
    plot_side_by_side(df_i, df_n, left_title, right_title, out_path=out_path)

def main():
    import argparse
    parser = argparse.ArgumentParser(description="Plot blue robot odometry from two ROS2 .db3 bags side by side.")
    parser.add_argument("--ideal", default="/home/cago/MMI_714_generative_models/homeworks/ros2_motion_noise/run_A_ideal/run_A_ideal_0.db3",
                        help="Path to ideal run .db3")
    parser.add_argument("--noisy", default="/home/cago/MMI_714_generative_models/homeworks/ros2_motion_noise/run_A_noisy/run_A_noisy_0.db3",
                        help="Path to noisy run .db3")
    parser.add_argument("--topic", default=None, help="Explicit odometry topic (overrides auto-detection)")
    parser.add_argument("--color", default="blue", help="Substring to find the blue robot topic (used if --topic not provided)")
    parser.add_argument("--out", default="/home/cago/MMI_714_generative_models/homeworks/ros2_motion_noise/outputs/blue_odom_side_by_side.png",
                        help="Output figure path")
    args = parser.parse_args()

    plot_blue_from_bags(args.ideal, args.noisy, topic=args.topic, color=args.color, out_path=args.out)

if __name__ == "__main__":
    main()

In [ ]:
# Set paths and optional topic. Run this cell to generate the plot.

ideal_bag = "/home/cago/MMI_714_generative_models/homeworks/ros2_motion_noise/run_A_ideal/run_A_ideal_0.db3"
noisy_bag = "/home/cago/MMI_714_generative_models/homeworks/ros2_motion_noise/run_A_noisy/run_A_noisy_0.db3"

# Inspect available topics if needed
print("Ideal bag topics:")
_ = list_topics(ideal_bag)
print("\nNoisy bag topics:")
_ = list_topics(noisy_bag)

# If auto-detection misses, set topic explicitly, e.g. "/blue/odom"
topic = None  # or "/blue/odom"
out_path = "/home/cago/MMI_714_generative_models/homeworks/ros2_motion_noise/outputs/blue_odom_side_by_side.png"

plot_blue_from_bags(ideal_bag, noisy_bag, topic=topic, color="blue", out_path=out_path)